# ESM HDF5 → JSON Pipeline

This project provides a **robust, restart‑safe pipeline** to download earthquake waveform records from the **European Strong Motion (ESM) database**, store them in **HDF5 format**

The pipeline is designed for **large-scale datasets (thousands of records)** and includes:

- Concurrent downloads
- Batch processing
- Crash‑safe resume capability
- Progress monitoring
- Robust dataframe normalization

In [1]:
# Imports / constants

from __future__ import annotations

import csv
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Optional, Tuple

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

G0 = 9.80665  # m/s^2 per g
HDF5_MAGIC = b"\x89HDF\r\n\x1a\n"


## 0) Robust loader + normalization for `example_df.csv` (2-row header pattern)

In [2]:
def _safe_str(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x).strip()


def load_example_df_robust(csv_fp: str | Path) -> pd.DataFrame:
    raw = pd.read_csv(csv_fp, header=None)

    cols = [("" if (c is None or (isinstance(c, float) and np.isnan(c))) else str(c)).strip()
            for c in raw.iloc[1].tolist()]

    seen = {}
    fixed = []
    for c in cols:
        base = c if c else "col"
        if base not in seen:
            seen[base] = 0
            fixed.append(base)
        else:
            seen[base] += 1
            fixed.append(f"{base}_{seen[base]}")

    df = raw.iloc[2:].copy()
    df.columns = fixed
    df = df.dropna(axis=1, how="all").reset_index(drop=True)

    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()

    df.columns = [str(c).strip() for c in df.columns]
    return df


def normalize_metadata_df(df: pd.DataFrame) -> pd.DataFrame:
    cols = list(df.columns)
    lower = {str(c).lower(): c for c in cols}

    def pick(exact: list[str], contains_any: list[str]) -> str | None:
        for e in exact:
            if e.lower() in lower:
                return lower[e.lower()]
        for c in cols:
            cl = str(c).lower()
            if any(s.lower() in cl for s in contains_any):
                return c
        return None

    event_col = pick(["event_id", "eventid"], ["event"])
    sta_col = pick(["station_code", "station"], ["station"])
    loc_col = pick(["location_code", "location", "station_location"], ["location"])

    if event_col is None or sta_col is None:
        raise ValueError(f"Could not find event/station columns. Available columns: {cols}")

    out = df.copy()
    out = out.rename(columns={event_col: "event_id", sta_col: "station_code"})
    if loc_col is not None:
        out = out.rename(columns={loc_col: "location_code"})
    else:
        out["location_code"] = ""

    for c in ["event_id", "station_code", "location_code"]:
        out[c] = out[c].map(_safe_str)

    out["_invalid_row"] = (out["event_id"] == "") | (out["station_code"] == "")
    return out


## 1) Concurrent downloader

In [3]:
def download_records_to_hdf5(
    df: pd.DataFrame,
    dst_dir: Path,
    *,
    event_id_col: str = "event_id",
    station_code_col: str = "station_code",
    location_code_col: str = "location_code",
    data_type: str = "ACC",
    batch_size: int = 500,
    max_workers: int = 4,
    timeout: Tuple[float, float] = (10.0, 60.0),
    sleep_between: float = 0.1,
    overwrite: bool = False,
    manifest_fp: Optional[Path] = None,
    total_retries: int = 2,
    backoff_factor: float = 0.5,
    show_progress: bool = True,
) -> list[Path]:
    def _ensure_hdf5_file(fp: Path) -> None:
        with open(fp, "rb") as f:
            if f.read(8) != HDF5_MAGIC:
                raise ValueError(f"Downloaded file is not valid HDF5: {fp}")

    def _build_url(event_id: str, station: str) -> str:
        from urllib.parse import urlencode
        qs = urlencode({"eventid": event_id, "station": station, "data-type": data_type})
        return f"https://esm-db.eu/esmws/eventdata/1/query?{qs}"

    def _make_session() -> requests.Session:
        session = requests.Session()
        retry = Retry(
            total=total_retries,
            connect=total_retries,
            read=total_retries,
            status=total_retries,
            backoff_factor=backoff_factor,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["GET"]),
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retry, pool_connections=max_workers * 4, pool_maxsize=max_workers * 4)
        session.mount("https://", adapter)
        session.mount("http://", adapter)
        return session

    def _progress_futures(futures, total: int, desc: str):
        if not show_progress:
            for fut in as_completed(futures):
                yield fut
            return
        try:
            from tqdm import tqdm  # type: ignore
            for fut in tqdm(as_completed(futures), total=total, desc=desc):
                yield fut
        except Exception:
            for i, fut in enumerate(as_completed(futures), start=1):
                if i == 1 or i % 50 == 0 or i == total:
                    print(f"{desc}: {i}/{total}")
                yield fut

    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    if manifest_fp is None:
        manifest_fp = dst_dir / "download_manifest.csv"
    manifest_fp = Path(manifest_fp)

    if event_id_col not in df.columns or station_code_col not in df.columns:
        raise ValueError(f"df must contain '{event_id_col}' and '{station_code_col}'")

    df_local = df.copy()
    if location_code_col not in df_local.columns:
        df_local[location_code_col] = ""

    for c in [event_id_col, station_code_col, location_code_col]:
        df_local[c] = df_local[c].map(_safe_str)

    df_local["record_id"] = df_local[event_id_col] + "__" + df_local[station_code_col] + "__" + df_local[location_code_col]

    manifest_cols = ["record_id","event_id","station_code","location_code","url","status","http_status","error_type","error_message","timestamp"]
    if not manifest_fp.exists():
        with open(manifest_fp, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=manifest_cols).writeheader()

    done: set[str] = set()
    if manifest_fp.exists() and not overwrite:
        try:
            m = pd.read_csv(manifest_fp, dtype=str).fillna("")
            done = set(m.loc[m["status"] == "downloaded", "record_id"].tolist())
        except Exception:
            done = set()

    session = _make_session()
    ok_files: list[Path] = []

    def _download_one(row_dict: dict[str, str]) -> dict[str, Any]:
        rec_id = row_dict["record_id"]
        event_id = row_dict[event_id_col]
        station_code = row_dict[station_code_col]
        loc_code = row_dict[location_code_col]
        dst_fp = dst_dir / f"{rec_id}.h5"

        if not event_id or not station_code:
            return {"record_id": rec_id,"event_id": event_id,"station_code": station_code,"location_code": loc_code,
                    "url": "", "status": "invalid_metadata","http_status":"","error_type":"ValueError",
                    "error_message":"Missing event_id or station_code; request skipped.","timestamp": pd.Timestamp.now("UTC").isoformat()}

        if not overwrite and (rec_id in done or dst_fp.exists()):
            return {"record_id": rec_id,"event_id": event_id,"station_code": station_code,"location_code": loc_code,
                    "url": "", "status": "skipped","http_status":"","error_type":"","error_message":"",
                    "timestamp": pd.Timestamp.now("UTC").isoformat()}

        url = _build_url(event_id, station_code)
        status, http_status, err_type, err_msg = "failed", "", "", ""

        try:
            with session.get(url, stream=True, timeout=timeout) as r:
                http_status = str(r.status_code)
                if r.status_code != 200:
                    raise requests.HTTPError(f"HTTP {r.status_code} for {url}")

                tmp = dst_fp.with_suffix(f".part.{os.getpid()}")
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                tmp.replace(dst_fp)

            _ensure_hdf5_file(dst_fp)
            status = "downloaded"
        except Exception as e:
            err_type = type(e).__name__
            err_msg = str(e)[:500]
            for p in dst_dir.glob(dst_fp.stem + ".part*"):
                try: p.unlink()
                except Exception: pass

        time.sleep(sleep_between)
        return {"record_id": rec_id,"event_id": event_id,"station_code": station_code,"location_code": loc_code,
                "url": url,"status": status,"http_status": http_status,"error_type": err_type,"error_message": err_msg,
                "timestamp": pd.Timestamp.now("UTC").isoformat()}

    n = len(df_local)
    for bstart in range(0, n, batch_size):
        bend = min(bstart + batch_size, n)
        batch = df_local.iloc[bstart:bend]
        if not overwrite and done:
            batch = batch[~batch["record_id"].isin(done)]
        rows = batch.to_dict(orient="records")
        if not rows:
            continue

        print(f"Batch {bstart+1}-{bend} (submitting {len(rows)} tasks) ...")

        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = [ex.submit(_download_one, r) for r in rows]
            with open(manifest_fp, "a", newline="", encoding="utf-8") as f:
                w = csv.DictWriter(f, fieldnames=manifest_cols)
                for fut in _progress_futures(futures, total=len(futures), desc="Downloading"):
                    out = fut.result()
                    w.writerow(out); f.flush()
                    if out["status"] == "downloaded":
                        done.add(out["record_id"])
                        ok_files.append(dst_dir / f"{out['record_id']}.h5")

    return ok_files


## Run the pipeline

In [4]:
# ---- Pipeline runner ----

ESM_FLATFILE_PATH = Path(r"C:\Users\clemettn\Documents\phd\data_raw\gm_flatfiles\ESM_flatfile_SA.csv")
CSV_PATH = Path(r"D:\ESM_hdf5_files\ESM_hdf5_files_to_download.csv")  # change to your real metadata CSV path

HDF5_DIR = Path(r"D:\ESM_hdf5_files")
# JSON_DIR = Path(r"C:\Users\clemettn\Documents\phd\data_processed\07_gm_records")
# JSON_SUBSET_DIR = Path("out_json_subset")

# Load the flatfile and resave as comma separated csv instead of a semicolon separated one.
df_esm_flat = pd.read_csv(ESM_FLATFILE_PATH, sep=";", dtype=str)
df_esm_flat = df_esm_flat[["event_id", "station_code", "location_code"]]
df_esm_flat = df_esm_flat.loc[df_esm_flat["location_code"] == "00", :]
df_esm_flat.columns = pd.MultiIndex.from_product([["metadata"], df_esm_flat.columns])
df_esm_flat.to_csv(CSV_PATH, index=False)

# Load + normalize
df_raw = load_example_df_robust(CSV_PATH)
df_norm = normalize_metadata_df(df_raw)

invalid = df_norm[df_norm["_invalid_row"]].copy()
if len(invalid) > 0:
    print(f"⚠️ Found {len(invalid)} invalid rows (missing event_id or station_code). They will be skipped for download.")

df = df_norm[~df_norm["_invalid_row"]].drop(columns=["_invalid_row"]).copy()
df = df.drop_duplicates(subset=["event_id", "station_code", "location_code"]).reset_index(drop=True)

print("Rows to process:", len(df))
display(df.head())

# Download (resume-safe)
downloaded = download_records_to_hdf5(
    df=df,
    dst_dir=HDF5_DIR,
    batch_size=500,
    max_workers=4,
    timeout=(10.0, 60.0),
    sleep_between=0.1,
    overwrite=False,
    manifest_fp=HDF5_DIR / "download_manifest.csv",
    total_retries=2,
    backoff_factor=0.5,
    show_progress=True,
)
print("HDF5 downloaded/present:", len(downloaded), f"out of {len(df)} requested")
print(f"{len(df) - len(downloaded)} records could not be downloaded.")

Rows to process: 22579


,event_id,station_code,location_code
0,AL-2014-0005,FIER,00
1,AL-2014-0005,KKS,00
2,AL-2014-0005,SDA,00
3,AL-2016-0001,DURR,00
4,AL-2016-0001,KBN,00


Batch 1-500 (submitting 500 tasks) ...


Downloading: 100%|██████████| 500/500 [01:23<00:00,  5.96it/s]


Batch 501-1000 (submitting 500 tasks) ...


...

Batch 22001-22500 (submitting 500 tasks) ...


Downloading: 100%|██████████| 500/500 [01:26<00:00,  5.81it/s]


Batch 22501-22579 (submitting 79 tasks) ...


Downloading: 100%|██████████| 79/79 [00:13<00:00,  5.72it/s]

HDF5 downloaded/present: 21688 out of 22579 requested
891 records could not be downloaded.
